# ListenBrainz dump → daily aggregates → local Postgres
This notebook:
1. Reads a **local** `listenbrainz-*-listens-*.tar.zst` dump file
2. Streams decompress + tar extraction (no full unzip needed)
3. Parses `.listens` JSON-lines records
4. Builds daily aggregates for tracks and artists
5. Upserts into a **local Postgres** database

In [16]:
## Install dependencies (run once per environment)
!pip -q install zstandard orjson psycopg2-binary pandas tqdm


python(23251) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


## 1) Configure paths + database connection

```bash
export PGHOST=localhost
export PGPORT=5432
export PGDATABASE=newsfeed
export PGUSER=newsfeed
export PGPASSWORD=newsfeed
```


In [17]:
import os
from pathlib import Path

# --- Set your dump path here ---
DUMP_PATH = Path("/Users/didiermunezero/Documents/NU/Junior/DE 300/de300-2026wi-munezero/Final Project/datadumps/musicbrainz/listenbrainz-listens-dump-2428-20260213-000003-incremental.tar.zst")

assert DUMP_PATH.exists(), f"Dump not found: {DUMP_PATH.resolve()}"

# --- Postgres connection via env vars (recommended) ---
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = int(os.getenv("PGPORT", "5432"))
PGDATABASE = os.getenv("PGDATABASE", "newsfeed")
PGUSER = os.getenv("PGUSER", "newsfeed")
PGPASSWORD = os.getenv("PGPASSWORD", "newsfeed")  # set via env

print("Dump:", DUMP_PATH.name)
print("Postgres:", f"{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")


Dump: listenbrainz-listens-dump-2428-20260213-000003-incremental.tar.zst
Postgres: newsfeed@localhost:5432/newsfeed


## 2) Create tables (local Postgres)
Creates the complete schema with stats tables.
Safe to re-run (`IF NOT EXISTS`) unless you changed the schema - in that case, run the drop cell above first.


**WARNING:** Run this cell to drop existing tables and start fresh (you'll lose existing data)

In [19]:
import psycopg2

# Drop tables in correct order (children before parents due to foreign keys)
drop_ddl = '''
DROP TABLE IF EXISTS track_daily_stats CASCADE;
DROP TABLE IF EXISTS artist_daily_stats CASCADE;
DROP TABLE IF EXISTS track_daily_listens CASCADE;
DROP TABLE IF EXISTS artist_daily_listens CASCADE;
DROP TABLE IF EXISTS track_info CASCADE;
DROP TABLE IF EXISTS artist_info CASCADE;
DROP TABLE IF EXISTS ingestion_state CASCADE;
'''

conn = psycopg2.connect(
    host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(drop_ddl)
conn.close()

print("All tables dropped. Run the next cell to recreate them.")

All tables dropped. Run the next cell to recreate them.


In [20]:
import psycopg2

ddl = '''
CREATE TABLE IF NOT EXISTS ingestion_state (
  id                INT PRIMARY KEY DEFAULT 1,
  last_dump_id      TEXT,
  last_dump_path    TEXT,
  loaded_at         TIMESTAMPTZ NOT NULL DEFAULT now(),
  CONSTRAINT singleton_row CHECK (id = 1)
);
INSERT INTO ingestion_state (id) VALUES (1)
ON CONFLICT (id) DO NOTHING;

CREATE TABLE IF NOT EXISTS artist_info (
  artist_mbid TEXT PRIMARY KEY,
  artist_name TEXT
);

CREATE TABLE IF NOT EXISTS track_info (
  recording_id TEXT PRIMARY KEY,
  track_name TEXT,
  artist_mbids TEXT[],
  release_name TEXT
);

CREATE TABLE IF NOT EXISTS artist_daily_listens (
  day DATE NOT NULL,
  artist_mbid TEXT NOT NULL,
  listen_count BIGINT NOT NULL,
  PRIMARY KEY (day, artist_mbid),
  FOREIGN KEY (artist_mbid) REFERENCES artist_info(artist_mbid)
);

CREATE TABLE IF NOT EXISTS track_daily_listens (
  day DATE NOT NULL,
  recording_id TEXT NOT NULL,
  listen_count BIGINT NOT NULL,
  PRIMARY KEY (day, recording_id),
  FOREIGN KEY (recording_id) REFERENCES track_info(recording_id)
);

CREATE TABLE IF NOT EXISTS artist_daily_stats (
  day DATE NOT NULL,
  artist_mbid TEXT NOT NULL,
  growth_percentile FLOAT,
  cumulative_listen_count BIGINT,
  listen_count_past_7_days BIGINT,
  listen_pctl_past_7_days FLOAT,
  listen_count_past_30_days BIGINT,
  listen_pctl_past_30_days FLOAT,
  PRIMARY KEY (day, artist_mbid),
  FOREIGN KEY (artist_mbid) REFERENCES artist_info(artist_mbid)
);

CREATE TABLE IF NOT EXISTS track_daily_stats (
  day DATE NOT NULL,
  recording_id TEXT NOT NULL,
  growth_percentile FLOAT,
  cumulative_listen_count BIGINT,
  listen_count_past_7_days BIGINT,
  listen_pctl_past_7_days FLOAT,
  listen_count_past_30_days BIGINT,
  listen_pctl_past_30_days FLOAT,
  PRIMARY KEY (day, recording_id),
  FOREIGN KEY (recording_id) REFERENCES track_info(recording_id)
);

CREATE INDEX IF NOT EXISTS idx_track_daily_day ON track_daily_listens(day);
CREATE INDEX IF NOT EXISTS idx_artist_daily_day ON artist_daily_listens(day);
CREATE INDEX IF NOT EXISTS idx_track_stats_day ON track_daily_stats(day);
CREATE INDEX IF NOT EXISTS idx_artist_stats_day ON artist_daily_stats(day);
'''

conn = psycopg2.connect(
    host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(ddl)
conn.close()

print("Tables ready.")


Tables ready.


## 3) Stream-parse the dump and build aggregates
Parses `.listens` files inside the tarball (one JSON object per line).
Handles common casing variations like `recording_mbid` vs `Recording_mbid`.

**Speed knob:** set env var `MAX_LINES` to cap records for a quick test.
Example: `export MAX_LINES=200000`


In [21]:
import tarfile
import zstandard as zstd
import orjson
from collections import defaultdict
from datetime import datetime, timezone
from tqdm import tqdm

def day_from_unix(ts: int) -> str:
    return datetime.fromtimestamp(ts, tz=timezone.utc).date().isoformat()

def get_any(d, *keys):
    if not isinstance(d, dict):
        return None
    for k in keys:
        if k in d:
            return d[k]
    return None

def iter_listens_lines_from_tar_zst(path: Path):
    """Yield (member_name, line_bytes) for each non-empty line in *.listens files inside tar.zst."""
    with path.open("rb") as fh:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(fh) as reader:
            with tarfile.open(fileobj=reader, mode="r|*") as tf:
                for member in tf:
                    if not member.isfile():
                        continue
                    name = member.name
                    if "/listens/" not in name or not name.endswith(".listens"):
                        continue
                    f = tf.extractfile(member)
                    if f is None:
                        continue
                    for line in f:
                        line = line.strip()
                        if line:
                            yield name, line

track_daily = defaultdict(int)    # (day, recording_id) -> count
artist_daily = defaultdict(int)   # (day, artist_mbid) -> count
track_info = {}   # recording_id -> (track_name, artist_mbids, release_name)
artist_info = {}  # artist_mbid -> artist_name (best-effort)

MAX_LINES = int(os.getenv("MAX_LINES", "0"))  # 0 = no limit

lines = 0
bad_json = 0
missing_ts = 0
missing_recording = 0
missing_artist = 0

for _, line in tqdm(iter_listens_lines_from_tar_zst(DUMP_PATH), desc="Parsing listens"):
    lines += 1
    if MAX_LINES and lines > MAX_LINES:
        break

    try:
        rec = orjson.loads(line)
    except Exception:
        bad_json += 1
        continue

    ts = rec.get("timestamp")
    if ts is None:
        missing_ts += 1
        continue

    day = day_from_unix(int(ts))

    tm = rec.get("track_metadata") or {}
    add = tm.get("additional_info") or {}

    artist_mbids = get_any(add, "artist_mbids") or []

    track_name = tm.get("track_name")
    release_name = tm.get("release_name")
    artist_name = tm.get("artist_name")

    # Create recording_id from artist_mbids and track_name
    recording_id = f"{'_'.join(artist_mbids)}_{track_name}"

    # Track daily listens
    track_daily[(day, recording_id)] += 1
    if recording_id not in track_info:
        track_info[recording_id] = (track_name, artist_mbids, release_name)

    # Artist daily listens
    if isinstance(artist_mbids, list):
        for ambid in artist_mbids:
            if not ambid:
                continue
            artist_daily[(day, ambid)] += 1
            if ambid not in artist_info and artist_name:
                artist_info[ambid] = artist_name
    else:
        if artist_mbids is None or artist_mbids == "" or artist_mbids == []:
            missing_artist += 1

    

summary = {
    "lines_parsed": lines,
    "bad_json": bad_json,
    "missing_timestamp": missing_ts,
    "missing_recording_mbid": missing_recording,
    "missing_artist_mbid": missing_artist,
    "unique_track_day_keys": len(track_daily),
    "unique_artist_day_keys": len(artist_daily),
    "unique_tracks": len({k[1] for k in track_daily.keys()}),
    "unique_artists": len({k[1] for k in artist_daily.keys()}),
}
summary


Parsing listens: 3168215it [00:36, 87434.65it/s] 


{'lines_parsed': 3168215,
 'bad_json': 0,
 'missing_timestamp': 0,
 'missing_recording_mbid': 0,
 'missing_artist_mbid': 0,
 'unique_track_day_keys': 2377730,
 'unique_artist_day_keys': 18238,
 'unique_tracks': 628306,
 'unique_artists': 17579}

## 4) Quick inspection with pandas


In [22]:
import pandas as pd

df_track_daily = pd.DataFrame(
    [{"day": day, "recording_id": rid, "listen_count": cnt}
     for (day, rid), cnt in track_daily.items()]
)

df_artist_daily = pd.DataFrame(
    [{"day": day, "artist_mbid": mbid, "listen_count": cnt}
     for (day, mbid), cnt in artist_daily.items()]
)

display(df_track_daily.head())
display(df_artist_daily.head())


,day,recording_id,listen_count
0,2026-02-11,_Lonely,1
1,2026-02-11,_All white,1
2,2026-02-11,_DUMM,2
3,2026-02-11,_Connect Connect,2
4,2026-02-11,_SIDE QUEST (RED BULL 64 BARS),2


,day,artist_mbid,listen_count
0,2026-02-12,8264722b-df00-467a-858e-5c97cda169c9,319
1,2026-02-12,8032cf05-d916-4b2a-9c53-6e75d4a24bd8,7
2,2026-02-12,83b46bf8-68fc-4159-bba3-060cb7754383,1
3,2026-02-12,05a6f249-029a-46aa-a0d2-8f14d71131de,1
4,2026-02-12,6def3fa9-836f-4b50-8815-b96cc92e63b8,29


In [23]:
# Top tracks / artists in this dump
display(df_track_daily.sort_values("listen_count", ascending=False).head(15))
display(df_artist_daily.sort_values("listen_count", ascending=False).head(15))


,day,recording_id,listen_count
56,2026-02-12,_Cool Gadget for your pc UNDER 10$ #shorts #ga...,1157
1492371,2020-08-05,_Pink Noise (Loopable),624
1491593,2020-07-03,_Pink Noise (Loopable),606
1491809,2020-07-11,_Pink Noise (Loopable),606
1491782,2020-07-10,_Pink Noise (Loopable),598
1491954,2020-07-17,_Pink Noise (Loopable),597
1492847,2020-08-31,_Pink Noise (Loopable),594
1491875,2020-07-14,_Pink Noise (Loopable),589
1491353,2020-06-24,_Pink Noise (Loopable),579
1712489,2026-02-12,b8a7c51f-362c-4dcb-a259-bc6e0095f0a6_2step,572


,day,artist_mbid,listen_count
32,2026-02-12,b8a7c51f-362c-4dcb-a259-bc6e0095f0a6,753
13334,2026-02-12,1adb0669-5886-43fe-81f8-fd00d62fadff,407
248,2026-02-12,f59c5520-5f46-4d2c-b2c4-822eabf53419,399
945,2026-02-12,89aa5ecb-59ad-46f5-b3eb-2d424e941f19,397
77,2026-02-12,a74b1b7f-71a5-4011-9441-d0b5e4122711,356
1704,2026-02-12,89ad4ac3-39f7-470e-963a-56509c546377,331
0,2026-02-12,8264722b-df00-467a-858e-5c97cda169c9,319
137,2026-02-12,e134b52f-2e9e-4734-9bc3-bea9648d1fa1,318
1261,2026-02-12,381086ea-f511-4aba-bdf9-71c753dc5077,313
523,2026-02-12,056e4f3e-d505-4dad-8ec1-d04f521cbb56,305


## 5) Bulk upsert into Postgres
Uses `execute_values` for performance.
Aggregates are upserted by adding to existing counts (so you can re-run multiple dumps).


In [24]:
import psycopg2
from psycopg2.extras import execute_values
import re

def chunked(iterable, n=50_000):
    buf = []
    for x in iterable:
        buf.append(x)
        if len(buf) >= n:
            yield buf
            buf = []
    if buf:
        yield buf

# Extract dump_id from filename (e.g., "listenbrainz-listens-dump-2428-...")
dump_id_match = re.search(r'dump-(\d+)-', DUMP_PATH.name)
dump_id = dump_id_match.group(1) if dump_id_match else None

conn = psycopg2.connect(
    host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
)
conn.autocommit = False

with conn, conn.cursor() as cur:
    # Upsert artist_info
    artist_rows = [(k, v) for k, v in artist_info.items()]
    for batch in chunked(artist_rows, n=20_000):
        execute_values(
            cur,
            '''
            INSERT INTO artist_info (artist_mbid, artist_name)
            VALUES %s
            ON CONFLICT (artist_mbid) DO UPDATE
            SET artist_name = COALESCE(EXCLUDED.artist_name, artist_info.artist_name)
            ''',
            batch
        )

    # Upsert track_info
    track_rows = [(k, v[0], v[1], v[2]) for k, v in track_info.items()]
    for batch in chunked(track_rows, n=20_000):
        execute_values(
            cur,
            '''
            INSERT INTO track_info (recording_id, track_name, artist_mbids, release_name)
            VALUES %s
            ON CONFLICT (recording_id) DO UPDATE
            SET track_name = COALESCE(EXCLUDED.track_name, track_info.track_name),
                artist_mbids = COALESCE(EXCLUDED.artist_mbids, track_info.artist_mbids),
                release_name = COALESCE(EXCLUDED.release_name, track_info.release_name)
            ''',
            batch
        )

    # Upsert track_daily_listens
    track_daily_rows = [(day, rid, cnt) for (day, rid), cnt in track_daily.items()]
    for batch in chunked(track_daily_rows, n=50_000):
        execute_values(
            cur,
            '''
            INSERT INTO track_daily_listens (day, recording_id, listen_count)
            VALUES %s
            ON CONFLICT (day, recording_id) DO UPDATE
            SET listen_count = track_daily_listens.listen_count + EXCLUDED.listen_count
            ''',
            batch
        )

    # Upsert artist_daily_listens
    artist_daily_rows = [(day, mbid, cnt) for (day, mbid), cnt in artist_daily.items()]
    for batch in chunked(artist_daily_rows, n=50_000):
        execute_values(
            cur,
            '''
            INSERT INTO artist_daily_listens (day, artist_mbid, listen_count)
            VALUES %s
            ON CONFLICT (day, artist_mbid) DO UPDATE
            SET listen_count = artist_daily_listens.listen_count + EXCLUDED.listen_count
            ''',
            batch
        )

    # Update ingestion_state
    cur.execute(
        '''
        UPDATE ingestion_state
        SET last_dump_id = %s, last_dump_path = %s, loaded_at = now()
        WHERE id = 1
        ''',
        (dump_id, str(DUMP_PATH))
    )

conn.close()
print("Upserts complete.")


Upserts complete.


## 6) Compute Daily Stats using Spark
Uses PySpark to compute rolling window aggregates, growth percentiles, and cumulative counts.
Spark is ideal for:
- Window functions (rolling 7/30 day aggregates)
- Percentile calculations across large datasets
- Complex analytical queries that would be slow in raw SQL

This reads from Postgres, computes stats in Spark, and writes back to the stats tables.

In [25]:
# Install PySpark if needed
!pip -q install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Create Spark session with Postgres JDBC support
spark = SparkSession.builder \
    .appName("ListenBrainz Stats") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.1") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Postgres JDBC connection properties
jdbc_url = f"jdbc:postgresql://{PGHOST}:{PGPORT}/{PGDATABASE}"
jdbc_props = {
    "user": PGUSER,
    "password": PGPASSWORD,
    "driver": "org.postgresql.Driver"
}

print("Spark session created:", spark.version)

python(32842) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(32860) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/20 23:11:08 WARN Utils: Your hostname, Didiers-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.220 instead (on interface en0)
26/02/20 23:11:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/homebrew/anaconda3/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/didiermunezero/.ivy2.5.2/cache
The jars for the packages stored in: /Users/didiermunezero/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-38274a3a-7c99-4dcb-971e-d070eaba3617;1.0
	confs: [default]
	

Spark session created: 4.1.1


In [27]:
# Read artist_daily_listens from Postgres
artist_df = spark.read.jdbc(
    url=jdbc_url,
    table="artist_daily_listens",
    properties=jdbc_props
)

print(f"Loaded {artist_df.count():,} artist daily records")

# Sort by artist and day to ensure proper window ordering
artist_df = artist_df.sort("artist_mbid", "day")

# Define window specifications using row-based windows (more reliable than range-based)
artist_window_unbounded = Window.partitionBy("artist_mbid").orderBy("day").rowsBetween(Window.unboundedPreceding, 0)
artist_window_7d = Window.partitionBy("artist_mbid").orderBy("day").rowsBetween(-6, 0)  # Last 7 days (including today)
artist_window_30d = Window.partitionBy("artist_mbid").orderBy("day").rowsBetween(-29, 0)  # Last 30 days (including today)

# Compute cumulative listen count
artist_df = artist_df.withColumn(
    "cumulative_listen_count",
    F.sum("listen_count").over(artist_window_unbounded)
)

# Compute rolling 7-day and 30-day listen counts
artist_df = artist_df.withColumn(
    "listen_count_past_7_days",
    F.sum("listen_count").over(artist_window_7d)
)
artist_df = artist_df.withColumn(
    "listen_count_past_30_days",
    F.sum("listen_count").over(artist_window_30d)
)

# Compute growth (today's count / cumulative from yesterday)
artist_window_lag = Window.partitionBy("artist_mbid").orderBy("day")
artist_df = artist_df.withColumn(
    "cumulative_yesterday",
    F.lag("cumulative_listen_count", 1).over(artist_window_lag)
)
artist_df = artist_df.withColumn(
    "growth_rate",
    F.when(F.col("cumulative_yesterday").isNotNull() & (F.col("cumulative_yesterday") > 0),
           F.col("listen_count") / F.col("cumulative_yesterday")
    ).otherwise(None)
)

# Compute percentiles for growth and rolling windows
artist_df = artist_df.withColumn(
    "growth_percentile",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("growth_rate").asc_nulls_first()))
)
artist_df = artist_df.withColumn(
    "listen_pctl_past_7_days",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("listen_count_past_7_days")))
)
artist_df = artist_df.withColumn(
    "listen_pctl_past_30_days",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("listen_count_past_30_days")))
)

# Select final columns for artist_daily_stats
artist_stats = artist_df.select(
    "day",
    "artist_mbid",
    "growth_percentile",
    "cumulative_listen_count",
    "listen_count_past_7_days",
    "listen_pctl_past_7_days",
    "listen_count_past_30_days",
    "listen_pctl_past_30_days"
)

# Write to Postgres (upsert by overwriting partition or full table)
artist_stats.write.jdbc(
    url=jdbc_url,
    table="artist_daily_stats",
    mode="overwrite",  # Can change to "append" if you want to handle duplicates differently
    properties=jdbc_props
)

print(f"Wrote {artist_stats.count():,} artist stats records")
artist_stats.show(10)

Loaded 18,238 artist daily records


Wrote 18,238 artist stats records


+----------+--------------------+-----------------+-----------------------+------------------------+-----------------------+-------------------------+------------------------+
|       day|         artist_mbid|growth_percentile|cumulative_listen_count|listen_count_past_7_days|listen_pctl_past_7_days|listen_count_past_30_days|listen_pctl_past_30_days|
+----------+--------------------+-----------------+-----------------------+------------------------+-----------------------+-------------------------+------------------------+
|2026-01-23|76b2e842-5e85-4c9...|              0.0|                      1|                       1|                    0.0|                        1|                     0.0|
|2026-01-26|31aa6f87-8d00-4ae...|              0.0|                      1|                       1|                    0.0|                        1|                     0.0|
|2026-02-03|5049ba10-e082-480...|              0.0|                      1|                       1|                    

In [28]:
# Read track_daily_listens from Postgres
track_df = spark.read.jdbc(
    url=jdbc_url,
    table="track_daily_listens",
    properties=jdbc_props
)

print(f"Loaded {track_df.count():,} track daily records")

# Sort by recording and day to ensure proper window ordering
track_df = track_df.sort("recording_id", "day")

# Define window specifications using row-based windows (more reliable than range-based)
track_window_unbounded = Window.partitionBy("recording_id").orderBy("day").rowsBetween(Window.unboundedPreceding, 0)
track_window_7d = Window.partitionBy("recording_id").orderBy("day").rowsBetween(-6, 0)  # Last 7 days (including today)
track_window_30d = Window.partitionBy("recording_id").orderBy("day").rowsBetween(-29, 0)  # Last 30 days (including today)

# Compute cumulative listen count
track_df = track_df.withColumn(
    "cumulative_listen_count",
    F.sum("listen_count").over(track_window_unbounded)
)

# Compute rolling 7-day and 30-day listen counts
track_df = track_df.withColumn(
    "listen_count_past_7_days",
    F.sum("listen_count").over(track_window_7d)
)
track_df = track_df.withColumn(
    "listen_count_past_30_days",
    F.sum("listen_count").over(track_window_30d)
)

# Compute growth (today's count / cumulative from yesterday)
track_window_lag = Window.partitionBy("recording_id").orderBy("day")
track_df = track_df.withColumn(
    "cumulative_yesterday",
    F.lag("cumulative_listen_count", 1).over(track_window_lag)
)
track_df = track_df.withColumn(
    "growth_rate",
    F.when(F.col("cumulative_yesterday").isNotNull() & (F.col("cumulative_yesterday") > 0),
           F.col("listen_count") / F.col("cumulative_yesterday")
    ).otherwise(None)
)

# Compute percentiles for growth and rolling windows
track_df = track_df.withColumn(
    "growth_percentile",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("growth_rate").asc_nulls_first()))
)
track_df = track_df.withColumn(
    "listen_pctl_past_7_days",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("listen_count_past_7_days")))
)
track_df = track_df.withColumn(
    "listen_pctl_past_30_days",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("listen_count_past_30_days")))
)

# Select final columns for track_daily_stats
track_stats = track_df.select(
    "day",
    "recording_id",
    "growth_percentile",
    "cumulative_listen_count",
    "listen_count_past_7_days",
    "listen_pctl_past_7_days",
    "listen_count_past_30_days",
    "listen_pctl_past_30_days"
)

# Write to Postgres
track_stats.write.jdbc(
    url=jdbc_url,
    table="track_daily_stats",
    mode="overwrite",
    properties=jdbc_props
)

print(f"Wrote {track_stats.count():,} track stats records")
track_stats.show(10)

Loaded 2,377,730 track daily records


Wrote 2,377,730 track stats records


+----------+--------------------+-----------------+-----------------------+------------------------+-----------------------+-------------------------+------------------------+
|       day|        recording_id|growth_percentile|cumulative_listen_count|listen_count_past_7_days|listen_pctl_past_7_days|listen_count_past_30_days|listen_pctl_past_30_days|
+----------+--------------------+-----------------+-----------------------+------------------------+-----------------------+-------------------------+------------------------+
|2005-06-06|  _Who Got the Funk?|              0.0|                      1|                       1|                    0.0|                        1|                     0.0|
|2005-06-06|      _Stay Positive|              0.0|                      1|                       1|                    0.0|                        1|                     0.0|
|2005-06-06|     _Who Dares Wins|              0.0|                      1|                       1|                    

In [29]:
# Stop Spark session
spark.stop()
print("Spark session stopped.")

Spark session stopped.


## 7) Sanity checks from SQL

In [30]:
import psycopg2

conn = psycopg2.connect(
    host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
)

with conn, conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM track_daily_listens;")
    track_ct = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM artist_daily_listens;")
    artist_ct = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM track_daily_stats;")
    track_stats_ct = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM artist_daily_stats;")
    artist_stats_ct = cur.fetchone()[0]
    cur.execute("SELECT * FROM ingestion_state WHERE id=1;")
    state = cur.fetchone()

conn.close()
print({
    "track_daily_listens_rows": track_ct, 
    "artist_daily_listens_rows": artist_ct,
    "track_daily_stats_rows": track_stats_ct,
    "artist_daily_stats_rows": artist_stats_ct
})
print("ingestion_state:", state)

{'track_daily_listens_rows': 2377730, 'artist_daily_listens_rows': 18238, 'track_daily_stats_rows': 2377730, 'artist_daily_stats_rows': 18238}
ingestion_state: (1, '2428', '/Users/didiermunezero/Documents/NU/Junior/DE 300/de300-2026wi-munezero/Final Project/datadumps/musicbrainz/listenbrainz-listens-dump-2428-20260213-000003-incremental.tar.zst', datetime.datetime(2026, 2, 21, 6, 59, 35, 256958, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))))


In [31]:
# Sample stats to verify computation
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
)

# Top artists by cumulative listens (most recent day)
artist_stats_query = '''
    SELECT a.day, ai.artist_name, a.cumulative_listen_count, 
           a.listen_count_past_7_days, a.growth_percentile,
           a.listen_pctl_past_7_days
    FROM artist_daily_stats a
    JOIN artist_info ai ON a.artist_mbid = ai.artist_mbid
    WHERE a.day = (SELECT MAX(day) FROM artist_daily_stats)
    ORDER BY a.cumulative_listen_count DESC
    LIMIT 10;
'''

# Top tracks by 7-day momentum  
track_stats_query = '''
    SELECT t.day, ti.track_name, t.listen_count_past_7_days,
           t.cumulative_listen_count, t.growth_percentile,
           t.listen_pctl_past_7_days
    FROM track_daily_stats t
    JOIN track_info ti ON t.recording_id = ti.recording_id
    WHERE t.day = (SELECT MAX(day) FROM track_daily_stats)
    ORDER BY t.listen_count_past_7_days DESC
    LIMIT 10;
'''

print("Top Artists (by cumulative listens):")
df_artist_stats = pd.read_sql(artist_stats_query, conn)
display(df_artist_stats)

print("\nTop Tracks (by 7-day listens):")
df_track_stats = pd.read_sql(track_stats_query, conn)
display(df_track_stats)

conn.close()

Top Artists (by cumulative listens):


/var/folders/70/kzs026sn2yj379rclfls772h0000gn/T/ipykernel_3405/749733715.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_artist_stats = pd.read_sql(artist_stats_query, conn)


,day,artist_name,cumulative_listen_count,listen_count_past_7_days,growth_percentile,listen_pctl_past_7_days
0,2026-02-13,Death Grips,98,98,0.0,1.0
1,2026-02-13,Charlie Parker,3,3,1.0,0.0



Top Tracks (by 7-day listens):


/var/folders/70/kzs026sn2yj379rclfls772h0000gn/T/ipykernel_3405/749733715.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_track_stats = pd.read_sql(track_stats_query, conn)


,day,track_name,listen_count_past_7_days,cumulative_listen_count,growth_percentile,listen_pctl_past_7_days
0,2026-02-13,Overload,20,43,0.666667,1.000000
1,2026-02-13,Cold Feet,11,18,0.777778,0.888889
2,2026-02-13,Give It To You,5,5,0.888889,0.777778
3,2026-02-13,White Woman’s Instagram,4,4,1.000000,0.666667
4,2026-02-13,Red Cross (Short Take 1),1,1,0.000000,0.000000
5,2026-02-13,Bubbles Buried In This Jungle,1,1,0.000000,0.000000
6,2026-02-13,I Used to Be a King,1,1,0.000000,0.000000
7,2026-02-13,Outcross (Original Mix),1,1,0.000000,0.000000
8,2026-02-13,Refracting,1,1,0.000000,0.000000
9,2026-02-13,Urge for Going,1,1,0.000000,0.000000
